# Gemma 4 Developer Agent — What Actually Moves the Leaderboard

*A field guide to the `swegemma` harness and the agent-config choices that matter — written after reading the harness spec end to end, forking the public baselines, and going through the discussion threads.*

**Who this is for:** anyone staring at the 0.12 wall wondering what to change next, or about to spend their one daily submission on a config they haven't validated. No magic here — just the mechanics, the honest gotchas, a diagram, quick-reference tables, and a **complete `analyzer + coder` bundle you can build on** (cell at the end writes and validates it for you).

**If you read nothing else:**
- The public wall is **~0.12**. Everyone who forks the top baseline lands there; ties break by who reached the score first, so a late fork won't medal on its own. Beating it is a real problem, not a config tweak.
- You get **~1 scored submission per day** — so *validate locally before you spend it* (section 6).
- Submissions are **declarative YAML only** (`agent.yaml`), compiled in a sandbox against closed registries. No `agent.py`.
- One base model per submission. The default `gemma-4-31b-it-qat-w4a16-ct` (INT4) is the pragmatic pick — fast startup, big KV headroom on the 4×L4 box.

Everything below is grounded in the competition's own `HARNESS_README.md`, which the next cell reads straight from the attached data.


In [ ]:
# Read the harness spec straight from the attached competition data, so this
# notebook stays honest to the source rather than to my memory of it.
import glob

def find_readme():
    for p in glob.glob("/kaggle/input/**/HARNESS_README.md", recursive=True):
        return p
    return None

readme = find_readme()
if readme:
    text = open(readme, encoding="utf-8").read()
    print(f"Found harness spec: {readme}  ({len(text):,} chars)\n")
    for line in text.splitlines():
        if line.startswith(("## ", "### ")):
            print(line)
else:
    text = ""
    print("HARNESS_README.md not attached. Add the competition as an input "
          "(Add Input -> the competition) to run the grounded cells.")


## The shape of the thing

Every task is scored across **two isolated sandboxes**: your compiled agent fixes the repo in **Container A**, then the patch is extracted and re-verified against a hidden test set in **Container B**. Your job is to produce the smallest patch that makes Container B's tests pass.

```
                        one task = one repo issue
                                   │
        ┌──────────────── Container A (agent sandbox) ─────────────────┐
        │  repo @ base_commit  ·  60 min / 500 turns / $ budget         │
        │                                                               │
        │     ┌───────────── your compiled ADK agent ─────────────┐     │
        │     │  swe_coder (edits, runs tests, submits)            │     │
        │     │      │ delegates exploration ▼ (AgentTool)         │     │
        │     │  code_analyzer  ──►  search_similar_code           │     │
        │     │  (read-only)         get_code_neighbors            │     │
        │     │                      get_code_subgraph, read_file  │     │
        │     └────────────────────────────────────────────────────┘    │
        │  submit_patch()  →  git add -N . && git diff HEAD              │
        └───────────────────────────────┬───────────────────────────────┘
                                         │  agent_patch
        ┌──────────────── Container B (verification) ──────────────────┐
        │  clean repo + HIDDEN tests  →  apply patch  →  PASS / FAIL     │
        └───────────────────────────────────────────────────────────────┘
```

The split at the top — a read-only **`code_analyzer`** wrapped as an `AgentTool` — is the one structural choice the top public baselines share. It keeps file dumps and graph traversals out of the coder's 32k context window.


## 1. The submission contract

`adk-submission` replaces standard ADK's `from_config()` (arbitrary Python imports) with a **sandboxed YAML compiler**. You declare agents, tools, sub-agents and generation params in YAML; the host resolves them against closed registries (`ToolRegistry`, `ModelRegistry`, `SkillRegistry`, `CallbackRegistry`). No Python entrypoint.

```
submission/
├── agent.yaml            # REQUIRED root (agent.yml / root_agent.yaml also accepted)
├── configs/sampling.yaml # optional, loaded via !include
├── prompts/system.md     # optional, loaded via !include (raw UTF-8)
├── sub_agents/*.yaml      # optional AgentTool / sub-agent configs
├── skills/<name>/SKILL.md # optional ADK skills
└── adapters/<name>/       # optional LoRA (adapter_config.json + adapter_model.safetensors)
```

Hard limit worth remembering: **total unpacked < 3 GiB** (adapters included), context ceiling **32,768 tokens** (prompt + reasoning + output). `!include` resolves `.md`/`.txt` as raw strings and `.yaml` recursively; `..` traversal, symlinks and absolute paths are blocked.


## 2. Budgets & limits — quick reference

| What | Value | Why it bites |
|---|---|---|
| Wall-clock / task | **60 min** | Container setup is *excluded*, but exploration still burns it |
| Max turns / task | **500** | `get_status()` to check what's left (free) |
| Single command timeout | **300 s** | a long `pytest` sweep can eat the whole thing |
| Command stdout | **5,000 chars** | pipe through `head`, use `grep -n` |
| `read_file` | **150 lines / 10,000 chars** | read narrow ranges, not whole files |
| Context window | **32,768 tokens** | the real constraint — guard it |
| `max_output_tokens` | 1–32,768 (dflt 16,384) | too big + long thoughts = truncated tool calls |
| `thinking_budget` | 1–32,768 (dflt 4,096) | keep modest; `thinking_level: NONE` to disable |
| Scored submissions | **~1 / day** | validate locally first (section 6) |

The model runs on **4× NVIDIA L4 (96 GB)**, `tensor_parallel_size=4`, `max_model_len=32768`. The default **`gemma-4-31b-it-qat-w4a16-ct`** (INT4, ~16–18 GB weights) leaves ~68 GB for KV cache — which is why long debugging trajectories don't fall over. bf16 31B/27B fit but leave the cache tight. **All agents must declare the same base model.**


## 3. The tools you actually get

The agent drives a SWE-bench-style loop over one checkout at `/workspace`:

**Execution / lifecycle**
- `run_command(cmd)` — shell in `/workspace` (300s, stdout ≤5,000 chars). `pytest`/`unittest` are *not* masked — targeted test runs work.
- `submit_patch()` — `git add -N . && git diff HEAD`; **free** and **ends the loop** next turn. Call it last.
- `get_status()` — remaining budget; free.

**Workspace files**
- `read_file(path, start, end)`, `edit_file(path, old, new)` (verbatim `old`, incremental), `write_file(path, content)` (new files).

**Code-intelligence graph** (present when the repo ships graph/embeddings)
- `search_similar_code(query, k)`, `get_code_neighbors(node)`, `get_code_subgraph(nodes)`.

The graph tools are why a small model can localize without crawling. Use them **before** opening files.


## 4. `agent.yaml`, decoded

`LlmAgent` (default `agent_class`) — the fields that matter: `name`, `model`, `description`, `instruction` (supports `{problem_description}` / `{hints}` session-state templating), `tools`, `skills`, `sub_agents`, and `generate_content_config`.

`generate_content_config` accepts sampling + `thinking_config`, but **`tools`, `system_instruction`, `http_options`, `safety_settings`, `response_schema` are rejected inside it by design** — that's the sandbox boundary. The end-of-notebook cell writes a full, documented `analyzer + coder` bundle; here's the root shape:

```yaml
name: swe_coder
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
generate_content_config: !include configs/sampling.yaml
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - agent_tool:                    # keep exploration out of the coder's context
      config_path: sub_agents/code_analyzer.yaml
      skip_summarization: true
```


## 5. What actually moves the leaderboard

None of this is exotic — it's discipline the small model can't supply for itself.

1. **Analyzer + coder split.** Delegate file/graph exploration to a read-only sub-agent wrapped as `agent_tool(skip_summarization: true)`, so intermediate output never lands in the coder's 32k window.
2. **Graph-first localization.** `search_similar_code` → `get_code_neighbors` → `get_code_subgraph` *before* opening files. The bug is usually at a hit or one hop away.
3. **Smallest correct edit.** Change only what the failing test needs. Every extra edit is a chance to break another test — and a wrong sprawling patch scores the same as none.
4. **`submit_patch()` is free and final.** Verify (`py_compile`, closest tests), clean up `/tmp` scratch, *then* submit — once.

### Failure modes → what to do

| Symptom | Cause | Fix |
|---|---|---|
| Turn wasted, `<\|tool_call>` cut off | thought block + big edit hit `max_output_tokens` | `thinking_budget` 4,096; **incremental** edits; don't re-explain in thought |
| Your patch contains `pytest.ini` / `conftest.py` | harness commits them into baseline; you touched them | never edit/delete them |
| Patch contains `/workspace/repro.py` | scratch left in the tree | put scratch in **`/tmp`**, or delete before submit |
| Empty patch, score 0 | ran out of budget exploring | `get_status()`; at <25% left, stop and fix → verify → submit |
| Submission *failed* (no score) | malformed bundle / bad model or asset | **validate locally first** (next section) |


## 6. Validate before you spend your one daily submission

You get ~**one scored submission per day**, and a malformed bundle just *fails* — a wasted day. Kaggle staff published a **"Gemma 4 Developer Agent Wheelhouse"** with the pinned packages (`google-adk`, `adk-submission`, `swegemma`) so you can compile-check offline (`compile_submission` + `validate_single_declared_model` + `discover_adapters`). Full local *eval* also needs the Gemma model (GPU), a few test deps missing from the public wheelhouse (`typing-inspection`, `inline-snapshot`, `dirty-equals`, `pytest-httpbin`), and `--sandbox subprocess` (Docker isn't available in Kaggle notebooks).

Even the **compile check** saves a day. The cell below is a dependency-free structural validator that catches the common mistakes without the wheelhouse.


In [ ]:
# Dependency-free structural validator — mirrors the host's hard checks.
import re, pathlib

ALLOWED = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
ROOTS = ["agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml"]

def validate(root):
    root = pathlib.Path(root)
    if not root.exists():
        print(f"(no dir at {root} — validator is reusable; point it at your bundle)"); return False
    problems = []
    roots = [n for n in ROOTS if (root / n).exists()]
    if len(roots) != 1: problems.append(f"need exactly one root config, found {roots}")
    models, includes = set(), []
    for y in root.rglob("*"):
        if y.suffix in (".yaml", ".yml"):
            txt = y.read_text(encoding="utf-8", errors="ignore")
            models |= set(re.findall(r"^\s*model:\s*['\"]?([^'\"\s#]+)", txt, re.M))
            for ref in re.findall(r"!include\s+(\S+)", txt) + re.findall(r"config_path:\s*(\S+)", txt):
                if not (y.parent / ref).resolve().exists(): includes.append(f"{y.name} -> {ref}")
    if includes: problems.append(f"unresolved include/config_path: {includes}")
    if len(models) != 1: problems.append(f"must declare exactly one base model, found {sorted(models)}")
    files = [p for p in root.rglob("*") if p.is_file()]
    bad_ext = [str(p.relative_to(root)) for p in files if p.suffix.lower() not in ALLOWED]
    if bad_ext: problems.append(f"disallowed extensions: {bad_ext}")
    total = sum(p.stat().st_size for p in files)
    if total > 3 * 1024**3: problems.append(f"bundle too big: {total} bytes")
    if problems:
        print("BLOCKING:"); [print("  -", p) for p in problems]; return False
    print(f"OK: 1 root, single model {sorted(models)}, {len(files)} files, {total} bytes"); return True


## 7. A complete `analyzer + coder` bundle you can build on

The cell below writes a full, documented submission to `/kaggle/working/submission/`, validates it with the checker above, and zips it to `submission.zip`. The prompts are deliberately short and original — the point is the *structure* and the *discipline*, which is what generalizes; tune the prompts to the repos you see.


In [ ]:
import os, zipfile, pathlib

BUNDLE = pathlib.Path("/kaggle/working/submission")
import shutil; shutil.rmtree(BUNDLE, ignore_errors=True); BUNDLE.mkdir(parents=True, exist_ok=True)

FILES = {
"agent.yaml": """
name: swe_coder
model: gemma-4-31b-it-qat-w4a16-ct
description: Fixes a repository issue with the smallest correct, verified patch.
instruction: !include prompts/system.md
generate_content_config: !include configs/sampling.yaml
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - agent_tool:
      config_path: sub_agents/code_analyzer.yaml
      skip_summarization: true
""",
"sub_agents/code_analyzer.yaml": """
name: code_analyzer
model: gemma-4-31b-it-qat-w4a16-ct
description: Read-only navigator. Given an issue, returns the exact location, root cause and fix plan.
instruction: !include ../prompts/analyzer.md
generate_content_config: !include ../configs/sampling.yaml
tools:
  - run_command
  - read_file
  - search_similar_code
  - get_code_neighbors
  - get_code_subgraph
""",
"configs/sampling.yaml": """
temperature: 0.2
top_p: 0.95
top_k: 40
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: false
""",
"prompts/system.md": """You are an autonomous engineer fixing ONE issue in the repo at /workspace. Ship the
smallest correct patch, then call submit_patch. You run on a small model, so be disciplined.

Workflow:
1. Understand: state the expected vs actual behaviour in one line.
2. Localize via the code_analyzer tool FIRST (it returns LOCATION / ROOT CAUSE / FIX PLAN);
   verify its claim by reading those exact lines.
3. Reproduce: a minimal script in /tmp (never /workspace), run it.
4. Fix: edit_file with a unique, verbatim old_string. One logical change per edit. Root cause,
   not symptom. Handle the edge cases the issue names.
5. Verify: python -m py_compile the file, rerun the repro, run the closest existing tests.
6. Submit: git status/diff, ensure only intended source changed, delete scratch, submit_patch once.

Hard rules: never edit tests, conftest.py, pytest.ini or CI files. Keep public APIs compatible.
The env is offline and pre-built — do not install packages. Anything left in /workspace is your
patch. Call get_status every ~8 tool calls; when <25% budget remains, go straight to fix -> verify
-> submit. An empty patch scores zero, so always leave a best-effort fix.
""",
"prompts/analyzer.md": """You are code_analyzer, a read-only navigator. You never modify files. Given an issue,
find exactly where it must be fixed.

Method:
1. Extract identifiers from the issue (function/class names, error text, paths).
2. search_similar_code with a symbol taken from the code (not a sentence); confirm every hit by
   reading it. Use get_code_neighbors to walk callers/callees to where behaviour diverges.
   get_code_subgraph to see how a few candidates connect. If a search returns nothing, change the
   term rather than repeating it.
3. Never guess line numbers — read the code.

Answer (<=250 words, nothing else):
LOCATION: <path>:<start>-<end> (<symbol>)
ROOT CAUSE: <1-2 sentences>
FIX PLAN: <concrete change>
RELATED: <other sites needing the same change, or none>
TESTS: <existing tests that exercise this>
CONFIDENCE: high | medium | low
""",
}

for rel, content in FILES.items():
    p = BUNDLE / rel; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content.strip() + "\n", encoding="utf-8")

ok = validate(BUNDLE)                     # the section-6 validator
zip_path = "/kaggle/working/submission.zip"
if os.path.exists(zip_path): os.remove(zip_path)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(BUNDLE.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(BUNDLE))
print("wrote submission.zip:", os.path.getsize(zip_path), "bytes ·", "VALID" if ok else "CHECK ABOVE")


## 8. The LoRA question (be careful here)

Fine-tuned LoRA adapters under `adapters/` are the obvious way to try to beat 0.12 — a small model taught the repos' patterns. But the honest status from the discussion: the **official `sample_submission`, which ships `adapters/main_lora` + `tool_lora`, currently FAILS to score** ("unhandled error while rerunning"). Adapters are served on the scoring side via vLLM `--lora-modules` (the agent's model is rewritten to `openai/<adapter-name>`), and no public submission has yet scored *with* an adapter.

So before investing days: confirm the adapter path scores end to end with a tiny rank-8 smoke test. A great LoRA the harness can't load is worth nothing. **Also unresolved:** whether distillation from external API models (to generate SFT trajectories) is allowed — organizers said they're conferring. Check the rules thread first.


## 9. Honest bottom line

- Forking the top baseline gives ~0.12 and ties a wall; late ties don't medal.
- The real edges: a **working** LoRA (once the adapter path is confirmed), smarter **context management** (the analyzer/AgentTool split, leaner prompts that leave more budget to fix), and careful **sampling / thinking-budget** tuning — each measured one change at a time, because the public LB is ~50% of the data and noisy.
- With one submission a day, **local validation is your force multiplier.**

If this saved you a wasted submission or made the harness click, an upvote helps others find it — and I'd genuinely like to hear in the comments what's moved *your* score.

*Built on the competition's `HARNESS_README.md` and the public baselines/discussion; corrections welcome.*
